# 01 · Өгөгдөл үүсгэх

**ANKO Bridge — Хятад дахь Монгол оюутны элсэлтийн шинжилгээ**

Энэ notebook нь синтетик өгөгдөл үүсгэдэг. `src/data_generator.py`-г дуудаж 5 CSV
болон SQLite файл бэлдэнэ:

- `data/universities.csv` — 50 жинхэнэ Хятадын их сургууль
- `data/programs.csv` — 200 хөтөлбөр (ANKO-гийн 4 төрөл: 6+6, 1+4, Fall Bachelor, Master/PhD)
- `data/students.csv` — 1,500 Монгол оюутан
- `data/applications.csv` — 2,250 өргөдөл
- `data/scholarships.csv` — ~500 тэтгэлэг

Бүх өгөгдлийн зургийг `seed=42` ашиглан тогтворжуулсан тул дахин үүсгэвэл яг
ижил утгатай гарна.


## Бодит ANKO Bridge-тэй уялдсан паттернууд

1. **2019 он** — ANKO Bridge байгуулагдсан, цөөн оюутан (~55).
2. **2020-2022** — COVID, хил хаалттай — өргөдөл бараг зогссон (10-17 жил тус бүрд).
3. **2023+** — хил нээгдэж, эрэлт нэг дор тэсэлсэн — 2025 онд peak (760 оюутан 6 сард).
4. **6+6 хөтөлбөрийн өсөлт** — 2019 онд ~35%, 2024 онд ~55% хүрсэн.
5. **Улаанбаатар-ийн давамгайлал** — нийт оюутны ~70%.
6. **HSK дүрэм** — Fall Bachelor-т HSK 4 + 120 оноо, хэлний бэлтгэлд (6+6, 1+4) ямар ч HSK хамаагүй.


In [1]:
# Өгөгдөл үүсгэх script-ийг дуудах
import subprocess
import sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, str(Path('..') / 'src' / 'data_generator.py')],
    capture_output=True, text=True, encoding='utf-8',
)
print(result.stdout)
if result.returncode != 0:
    print("ALDAA:", result.stderr)



ANKO Bridge — синтетик өгөгдөл үүсгэж байна (seed=42)
  ✓ universities.csv  (50 мөр)
  ✓ programs.csv      (200 мөр)
  ✓ students.csv      (1500 мөр)
  ✓ applications.csv  (2250 мөр)
  ✓ scholarships.csv  (516 мөр)
  ✓ anko.db           (anko.db)

──────────────────────────────────────────────────────────────────────
  ЖИЛЭЭР ӨРГӨДӨЛ (тоо)
──────────────────────────────────────────────────────────────────────
  2019:   53  ███
  2020:   31  ██
  2021:   13  
  2022:   37  ██
  2023:  187  ████████████
  2024:  558  █████████████████████████████████████
  2025: 1371  ███████████████████████████████████████████████████████████████████████████████████████████

──────────────────────────────────────────────────────────────────────
  АNKO ХӨТӨЛБӨРИЙН ЭЗЛЭХ ХУВЬ ЖИЛЭЭР (%)
──────────────────────────────────────────────────────────────────────
anko_program_type   1+4   6+6  Fall Bachelor  Master/PhD
year                                                    
2019               34.0  50.9       

In [2]:
# CSV-уудыг ачааллаад мөрийн тоог шалгая
import pandas as pd
from pathlib import Path

DATA = Path('..') / 'data'
for name in ['universities', 'programs', 'students', 'applications', 'scholarships']:
    df = pd.read_csv(DATA / f'{name}.csv')
    print(f'{name:<15} {len(df):>5} мөр × {len(df.columns):>2} багана')


universities       50 мөр ×  8 багана
programs          200 мөр × 11 багана
students         1500 мөр × 14 багана
applications     2250 мөр × 11 багана
scholarships      516 мөр ×  6 багана


In [3]:
# Жишээ мөрүүд — students.csv
students = pd.read_csv(DATA / 'students.csv')
students.head(8)


,student_id,name,birth_year,gender,aimag,district,school_type,gpa_percent,hsk_level,hsk_score,family_income_mnt,register_date,english_level,is_high_school_senior
0,1,Балдан Энхтайван,2001,M,Увс,NaN,Public,75.4,5,183,1526826,2019-08-14,Basic,True
1,2,Содном Цэрэн,2001,M,Сүхбаатар,NaN,Public,86.7,3,220,2555084,2019-07-20,No English,True
2,3,Энхболд Цэрэн,2001,M,Улаанбаатар,Хан-Уул,Private,86.8,4,200,3912988,2019-09-27,Intermediate,True
3,4,Дамдин Маралмаа,2002,F,Улаанбаатар,Багануур,Public,82.7,5,188,1929822,2019-07-30,Basic,True
4,5,Дашням Энхбат,2001,M,Улаанбаатар,Сүхбаатар,Private,92.4,5,209,2198967,2019-01-07,Intermediate,True
5,6,Лхагвасүрэн Идэрбат,2001,M,Улаанбаатар,Сонгинохайрхан,Public,79.6,5,196,2652721,2019-02-24,No English,False
6,7,Жамьян Цогт,2001,M,Улаанбаатар,Сонгинохайрхан,Public,72.9,4,127,3556487,2019-03-07,Basic,True
7,8,Бямбаа Мөнхтөр,1998,M,Дархан-Уул,NaN,Public,72.7,4,169,3946793,2019-06-12,Basic,False


In [4]:
# Жишээ мөрүүд — applications.csv
applications = pd.read_csv(DATA / 'applications.csv')
applications.head(8)


,app_id,student_id,uni_id,program_id,anko_program_type,apply_date,status,decision_days,visa_status,enrollment_date,agent_score
0,1,3,35,137,1+4,2020-01-30,Accepted,30,Rejected,NaN,45
1,2,3,21,82,Fall Bachelor,2019-09-28,Accepted,68,Approved,NaN,31
2,3,4,11,42,6+6,2019-09-23,Enrolled,45,Approved,2020-02-15,36
3,4,6,4,13,6+6,2019-06-12,Enrolled,53,Approved,2019-11-02,36
4,5,6,7,25,1+4,2019-04-15,Withdrawn,64,NaN,NaN,46
5,6,7,6,23,6+6,2019-05-25,Rejected,41,NaN,NaN,45
6,7,7,35,139,Fall Bachelor,2019-10-21,Rejected,32,NaN,NaN,39
7,8,7,1,3,6+6,2019-03-21,Enrolled,35,Approved,2019-07-24,26


## Дүгнэлт

Өгөгдөл амжилттай үүсэв. Дараагийн notebook (`02_eda.ipynb`) нь exploratory data
analysis хийнэ.
